### Example Exploratory Notebook

Use this notebook to explore the data generated by the pipeline in your preferred programming language.

**Note**: This notebook is not executed as part of the pipeline.

In [0]:
import dlt
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
@dlt.view
def silver_employee_cdf():
    df = spark.readStream.table("LIVE.bronze_employee_cdf")
 
    df = df.withColumn("Employee_ID", col("Employee_ID").cast("string"))
    df = df.withColumn("Performance_Score", col("Performance_Score").cast("int"))
    df = df.withColumn("Monthly_Salary", col("Monthly_Salary").cast("double"))
    df = df.withColumn("Name", trim(col("Name")))
    df = df.withColumn("Department", trim(col("Department")))
 
    return df

In [0]:
dlt.create_streaming_table("gold_employee_cdf_streaming_type1")
 
dlt.apply_changes(
    target = "gold_employee_cdf_streaming_type1",
    source = "silver_employee_cdf",
    keys = ["Employee_ID"],
    sequence_by = struct("Employee_ID", "_commit_timestamp"),
    ignore_null_updates = True,
    apply_as_deletes = expr("_change_type = 'delete'"),
    apply_as_truncates = expr("_change_type = 'truncate'"),
    stored_as_scd_type = 1
)